# Canopy Race Engineering Challenge: Hypercar Edition

Judging this contest:
- Select the local folder which contains all of the job files that we want to assess
- Create a worksheet to run the studies
- For each job in the study
    - Upload all of the jobs in the folder into a new row in the worksheet
    - Run the jobs in the worksheet
    - Jobs might be incorrectly formatted - populate from the default row first, and then overwrite
- Download all of the job scalar results from every row
    - Associate the job scalar results with the job file
    - Check for illegal config changes
    - Check for illegal constraint values
    - Add the lap time to the array of lap times
- Plot the lap times indexed by config name
    - Highlight illegal configs in one colour
    - Highlight constraint excess in another colour

In [ ]:
import canopy
import logging
import nest_asyncio

logging.basicConfig(level=logging.INFO)
nest_asyncio.apply()
authentication_data = canopy.prompt_for_authentication()
session = canopy.Session(authentication_data)

In [ ]:
from pathlib import Path

source_folder = 'ENTER_JOB_SUBMISSION_FOLDER'
default_job = 'default.json'
user_maths = 'ENTER_USER_MATHS_JSON_PATH'

json_names = [path.name for path in Path(source_folder).glob("*.json")]
print("Source folder contains " + str(len(json_names)) + " jsons")

## Create the Worksheet

In [ ]:
import json
from pathlib import Path

from canopy.openapi import (
    ConfigReferenceTenant,
    ConfigResolvedLabelsReference,
    StudyResolvedLabelsReference,
    WorksheetApi,
    WorksheetConfig,
    WorksheetPutWorksheetRequest,
    WorksheetRow,
    WorksheetRowStudy,
    UpdatedWorksheetDataOutline,
)

CONFIG_TYPES = ("car", "track", "weather", "userMaths", "constraints")


def load_job_json(job_file):
    path = Path(job_file)
    if not path.is_absolute():
        path = Path(source_folder) / path
    with path.open(encoding="utf-8") as file:
        job = json.load(file)
    if not isinstance(job.get("simConfig"), dict) and not isinstance(job.get("config"), dict):
        raise ValueError(f"{path} must contain a 'simConfig' or 'config' object")
    return job


def load_user_maths_config(user_maths_file):
    path = Path(user_maths_file)
    if not path.is_absolute():
        path = Path(source_folder) / path
    with path.open(encoding="utf-8") as file:
        user_maths_data = json.load(file)
    config = user_maths_data.get("config")
    if not isinstance(config, dict):
        raise ValueError(f"{path} must contain a 'config' object")
    return config


def worksheet_update_request(worksheet, rows):
    return WorksheetPutWorksheetRequest(
        name=worksheet.name,
        properties=worksheet.properties,
        outline=UpdatedWorksheetDataOutline(
            rows=rows,
            label_definitions=worksheet.outline.label_definitions,
        ),
        notes=worksheet.notes,
    )


async def clear_worksheet(session, worksheet_id):
    session.authentication.authenticate()
    tenant_id = session.authentication.tenant_id
    worksheet_api = WorksheetApi(session.async_client)
    result = await worksheet_api.worksheet_get_worksheet(tenant_id, worksheet_id)
    worksheet = result.worksheet
    await worksheet_api.worksheet_put_worksheet(
        tenant_id,
        worksheet_id,
        worksheet_update_request(worksheet, []),
    )
    return worksheet_id


async def upload_job_row(session, worksheet_id, job_file, default_job_file):
    job = load_job_json(job_file)
    default_job = load_job_json(default_job_file)
    default_sim_config = dict(default_job.get("simConfig", {}))
    sim_config = dict(default_sim_config)

    if isinstance(job.get("simConfig"), dict):
        sim_config.update(job["simConfig"])
    else:
        sim_config["car"] = job["config"]

    if default_sim_config.get("userMaths") is not None:
        sim_config["userMaths"] = default_sim_config["userMaths"]
    else:
        sim_config["userMaths"] = load_user_maths_config(user_maths)
    row_name = Path(job_file).stem

    session.authentication.authenticate()
    tenant_id = session.authentication.tenant_id
    worksheet_configs = []
    for config_type in CONFIG_TYPES:
        config_data = sim_config.get(config_type)
        if config_data is None:
            continue
        config_id = await canopy.create_config(
            session=session,
            config_type=config_type,
            name=f"{row_name} - {config_type}",
            config_data=config_data,
        )
        worksheet_configs.append(
            WorksheetConfig(
                config_type=config_type,
                reference=ConfigResolvedLabelsReference(
                    tenant=ConfigReferenceTenant(
                        tenant_id=tenant_id,
                        target_id=config_id,
                    )
                ),
                inherit_reference=False,
            )
        )

    if not worksheet_configs:
        raise ValueError(f"No supported simConfig components found in {job_file}")

    study_id = await canopy.create_study(
        session=session,
        study_type="dynamicLap",
        study_name=row_name,
        inputs=[config.reference.tenant.target_id for config in worksheet_configs],
        sim_version=session.tenant_sim_version.get(),
    )

    worksheet_api = WorksheetApi(session.async_client)
    result = await worksheet_api.worksheet_get_worksheet(tenant_id, worksheet_id)
    worksheet = result.worksheet
    rows = list(worksheet.outline.rows or [])
    rows.append(
        WorksheetRow(
            name=row_name,
            configs=worksheet_configs,
            study=WorksheetRowStudy(
                reference=StudyResolvedLabelsReference(
                    tenant_id=tenant_id,
                    target_id=study_id,
                )
            ),
        )
    )
    await worksheet_api.worksheet_put_worksheet(
        tenant_id,
        worksheet_id,
        worksheet_update_request(worksheet, rows),
    )
    return row_name

#### NOTE: Your worksheet will be cleared and overwritten by this cell

In [ ]:
worksheet_id = 'ENTER_WORKSHEET_ID'

# Start with a clean worksheet, then add defaults as the first row.
await clear_worksheet(session, worksheet_id)
await upload_job_row(session, worksheet_id, default_job, default_job)

# Each submission inherits any omitted config components from default.json.
for job_file in sorted(json_names):
    if Path(job_file).name == Path(default_job).name:
        continue
    await upload_job_row(session, worksheet_id, job_file, default_job)

In [ ]:
import math

from canopy.openapi import WorksheetApi

# Limits from the challenge notes. Values are checked in the stated units.
SCALAR_LIMITS = {
    "RaceEng_EBottoming": {"max": 5.0e5},
    "RaceEng_UndersteerT1": {"min": 6e-3},
    "RaceEng_UndersteerT13": {"min": 15e-3},
    "RaceEng_ETyreWearFL": {"max": 9.25e5},
    "RaceEng_ETyreWearRL": {"max": 8.5e5},
}

# Config values are converted from their stored SI units before checking.
CONFIG_LIMITS = {
    "hRideFSetup": {
        "path": "chassis.hRideFSetup",
        "min": 0.010,
        "max": 0.045
    },
    "hRideRSetup": {
        "path": "chassis.hRideRSetup",
        "min": 0.050,
        "max": 0.100
    },
    "aFlapF": {
        "path": "aero.flapAngles.aFlapF",
        "min": 10 / 180 * math.pi,
        "max": 45 / 180 * math.pi,
    },
    "kTorsionBar (front)": {
        "path": "suspension.front.internal.torsionBar.kTorsionBar",
        "min": 2500,
        "max": 20000,
    },
    "kTriSpring (rear)": {
        "path": "suspension.rear.internal.triSpring.kSpring",
        "min": 200000,
        "max": 20000000
    },
}


def value_at_config_path(config_data, path):
    value = config_data
    for part in path.split("."):
        if not isinstance(value, dict) or part not in value:
            return None
        value = value[part]
    return value


session.authentication.authenticate()
tenant_id = session.authentication.tenant_id
worksheet_api = WorksheetApi(session.async_client)
worksheet_response = await worksheet_api.worksheet_get_worksheet(tenant_id, worksheet_id)
worksheet_rows = list(worksheet_response.worksheet.outline.rows or [])

# Each row maps back to the job filename through the row name used during upload.
row_results = {}
constraint_violations = {}
illegal_config_changes = {}
analysis_warnings = {}

for row in worksheet_rows:
    row_name = row.name
    row_results[row_name] = {
        "job_file": f"{row_name}.json",
        "tLapTotal": [],
        "scalar_results": [],
    }
    analysis_warnings[row_name] = []

    if row.study is None or row.study.reference is None:
        analysis_warnings[row_name].append("Worksheet row has no study reference")
        continue

    study_id = row.study.reference.target_id
    study = await canopy.load_study(
        session=session,
        study_id=study_id,
        sim_type="DynamicLap",
        include_job_metadata=True,
        include_job_scalar_results=True,
    )

    if not study.jobs:
        analysis_warnings[row_name].append("DynamicLap study has no downloadable job results")
        continue

    row_constraint_violations = []
    for job_index, job in enumerate(study.jobs):
        scalar_data = dict(job.scalar_data or {})
        lap_time = scalar_data.get("tLapTotal")
        row_results[row_name]["tLapTotal"].append(lap_time)
        row_results[row_name]["scalar_results"].append(scalar_data)

        for scalar_name, limits in SCALAR_LIMITS.items():
            if scalar_name not in scalar_data:
                analysis_warnings[row_name].append(
                    f"Job {job_index}: missing scalar output {scalar_name}"
                )
                continue
            value = scalar_data[scalar_name]
            if ("min" in limits and value <= limits["min"]) or (
                "max" in limits and value >= limits["max"]
            ):
                row_constraint_violations.append(
                    {
                        "job_index": job_index,
                        "scalar": scalar_name,
                        "value": float(value),
                        "limits": limits,
                    }
                )

    if row_constraint_violations:
        constraint_violations[row_name] = row_constraint_violations

    car_config_reference = next(
        (
            config.reference.tenant.target_id
            for config in (row.configs or [])
            if config.config_type == "car" and config.reference is not None
        ),
        None,
    )
    if car_config_reference is None:
        analysis_warnings[row_name].append("Worksheet row has no car config reference")
        continue

    car_config_result = await canopy.load_config(session, car_config_reference)
    car_config_data = car_config_result.document.data
    row_config_violations = []

    for parameter, limits in CONFIG_LIMITS.items():
        raw_value = value_at_config_path(car_config_data, limits["path"])
        if raw_value is None:
            analysis_warnings[row_name].append(
                f"Car config is missing {parameter} at {limits['path']}"
            )
            continue
        if raw_value < limits["min"] or raw_value > limits["max"]:
            row_config_violations.append(
                {
                    "parameter": parameter,
                    "value": raw_value,
                    "allowed_range": (limits["min"], limits["max"]),
                }
            )

    if row_config_violations:
        illegal_config_changes[row_name] = row_config_violations

print(f"Analyzed {len(row_results)} worksheet rows.")
print(f"Rows with scalar constraint violations: {len(constraint_violations)}")
print(f"Rows with out-of-range config values: {len(illegal_config_changes)}")
print(f"Rows with missing analysis data: {sum(bool(warnings) for warnings in analysis_warnings.values())}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

status_colors = {
    "legal": "#3A7D44",
    "constraint": "#E69F00",
    "config": "#D1495B",
    "both": "#7651A8",
}

row_names = list(row_results)
fig, ax = plt.subplots(figsize=(max(10, len(row_names) * 0.45), 6))

for row_index, row_name in enumerate(row_names):
    has_config_violation = bool(illegal_config_changes.get(row_name))
    has_constraint_violation = bool(constraint_violations.get(row_name))
    if has_config_violation and has_constraint_violation:
        status = "both"
    elif has_config_violation:
        status = "config"
    elif has_constraint_violation:
        status = "constraint"
    else:
        status = "legal"

    lap_times = row_results[row_name]["tLapTotal"]
    valid_lap_times = [float(value) for value in lap_times if value is not None]
    if not valid_lap_times:
        continue

    # Small deterministic offsets keep multiple jobs in one row visible.
    offsets = [
        (job_index - (len(valid_lap_times) - 1) / 2) * 0.08
        for job_index in range(len(valid_lap_times))
    ]
    ax.scatter(
        [row_index + offset for offset in offsets],
        valid_lap_times,
        color=status_colors[status],
        s=58,
        edgecolors="white",
        linewidths=0.7,
        zorder=3,
    )

ax.set_xticks(range(len(row_names)))
ax.set_xticklabels(row_names, rotation=60, ha="right")
ax.set_xlabel("Worksheet row / config name")
ax.set_ylabel("Lap time (s)")
ax.set_title("DynamicLap times and legality by config")
ax.grid(axis="y", alpha=0.25)
ax.legend(
    handles=[
        Patch(facecolor=status_colors["legal"], label="Legal"),
        Patch(facecolor=status_colors["constraint"], label="Scalar constraint excess"),
        Patch(facecolor=status_colors["config"], label="Illegal config"),
        Patch(facecolor=status_colors["both"], label="Both violation types"),
    ],
    frameon=False,
)
fig.tight_layout()
plt.show()

In [ ]:
for worksheet_row in worksheet_rows:
    row_name = worksheet_row.name
    row_data = row_results.get(row_name, {})
    lap_times = row_data.get("tLapTotal", [])

    car_config_id = next(
        (
            config.reference.tenant.target_id
            for config in (worksheet_row.configs or [])
            if config.config_type == "car"
            and config.reference is not None
            and config.reference.tenant is not None
        ),
        None,
    )

    print(f"\n{'=' * 72}\n{row_name}")
    print(f"Lap time(s): {lap_times or 'No results'}")

    if car_config_id is None:
        print("Car configuration: not available")
    else:
        car_config = await canopy.load_config(session, car_config_id)
        car_data = car_config.document.data

        print("Car settings:")
        for parameter, limits in CONFIG_LIMITS.items():
            value = value_at_config_path(car_data, limits["path"])
            if value is None:
                print(f"  {parameter}: missing")
            else:
                print(f"  {parameter}: {value}")

    scalar_violations = constraint_violations.get(row_name, [])
    print("Scalar constraint violations:")
    if scalar_violations:
        for violation in scalar_violations:
            print(
                f"  Job {violation['job_index']}, {violation['scalar']}: "
                f"{violation['value']} (limits: {violation['limits']})"
            )
    else:
        print("  None")

    config_violations = illegal_config_changes.get(row_name, [])
    print("Illegal configuration values:")
    if config_violations:
        for violation in config_violations:
            lower, upper = violation["allowed_range"]
            print(
                f"  {violation['parameter']}: {violation['value']} "
                f"(allowed range: {lower} to {upper})"
            )
    else:
        print("  None")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

job_names = list(row_results)
job_colors = {
    job_name: plt.get_cmap("tab20")(index % 20)
    for index, job_name in enumerate(job_names)
}

config_values_by_job = {}
for worksheet_row in worksheet_rows:
    row_name = worksheet_row.name
    car_config_reference = next(
        (
            config.reference.tenant.target_id
            for config in (worksheet_row.configs or [])
            if config.config_type == "car"
            and config.reference is not None
            and config.reference.tenant is not None
        ),
        None,
    )
    if car_config_reference is None:
        continue

    loaded_car_config = await canopy.load_config(session, car_config_reference)
    car_data = loaded_car_config.document.data
    config_values_by_job[row_name] = {
        parameter: value_at_config_path(car_data, limits["path"])
        for parameter, limits in CONFIG_LIMITS.items()
    }

scalar_aliases = {"RaceEng_UndersteerT1": ("RaceEng_UndersteerT1Entry",)}
job_points = []
for job_name, row_data in row_results.items():
    for job_index, scalar_data in enumerate(row_data["scalar_results"]):
        lap_time = scalar_data.get("tLapTotal")
        if lap_time is None:
            continue
        job_points.append(
            {
                "job_name": job_name,
                "job_index": job_index,
                "lap_time": float(lap_time),
                "scalar_data": scalar_data,
            }
        )


def plot_parameter_scatter(parameter_names, get_x_value, title, x_label):
    column_count = 2
    row_count = (len(parameter_names) + column_count - 1) // column_count
    figure, axes = plt.subplots(
        row_count,
        column_count,
        figsize=(14, 4 * row_count),
        squeeze=False,
    )
    figure.suptitle(title, fontsize=15)

    for axis, parameter_name in zip(axes.flat, parameter_names):
        for point in job_points:
            x_value = get_x_value(point, parameter_name)
            if x_value is None:
                continue
            axis.scatter(
                float(x_value),
                point["lap_time"],
                color=job_colors[point["job_name"]],
                s=58,
                edgecolors="white",
                linewidths=0.7,
                zorder=3,
            )
        axis.set_title(parameter_name)
        axis.set_xlabel(x_label(parameter_name))
        axis.set_ylabel("Lap time (s)")
        axis.grid(alpha=0.25)

    for axis in axes.flat[len(parameter_names):]:
        axis.set_visible(False)

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=job_colors[job_name],
            markeredgecolor="white",
            markersize=8,
            label=job_name,
        )
        for job_name in job_names
    ]
    figure.legend(
        handles=legend_handles,
        title="Job / worksheet row",
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        frameon=False,
    )
    figure.tight_layout(rect=(0, 0, 0.82, 0.96))
    plt.show()


plot_parameter_scatter(
    list(CONFIG_LIMITS),
    lambda point, parameter: config_values_by_job.get(point["job_name"], {}).get(parameter),
    "Lap time vs setup parameters",
    lambda parameter: f"{parameter} (stored config units)",
)
plot_parameter_scatter(
    list(SCALAR_LIMITS),
    lambda point, parameter: point["scalar_data"].get(parameter)
    if point["scalar_data"].get(parameter) is not None
    else next(
        (
            point["scalar_data"][alias]
            for alias in scalar_aliases.get(parameter, ())
            if point["scalar_data"].get(alias) is not None
        ),
        None,
    ),
    "Lap time vs constraint outputs",
    lambda parameter: parameter,
)